# 02 Preprocessing validation

**Question.** Does the input pipeline produce the documented tensors (crop, [-1, 1] range, channels, dtype, shape)?

**Type.** Validation of conventions in docs/PREPROCESSING_AND_WEIGHTS.md. Synthetic checks run always; a real-image check runs only if `USE_REAL_IMAGE = True`.

In [ ]:
import json, sys
from pathlib import Path
REPO = Path.cwd()
while not (REPO / "pyproject.toml").exists() and REPO != REPO.parent:
    REPO = REPO.parent
import os; os.chdir(REPO)
import numpy as np
import torch
from ct_iqa.preprocessing.transforms import preprocess_ct_image
from ct_iqa.config.schema import ModelConfig
from ct_iqa.models.registry import build_model

In [ ]:
# SYNTHETIC: ramp image 512x512 in [0,1]
x = np.tile(np.linspace(0, 1, 512, dtype=np.float32), (512, 1))
y = preprocess_ct_image(x, 224)
assert y.shape == (224, 224) and y.dtype == np.float32
assert np.isclose(y.min(), 2 * x[0, 144] - 1) and np.isclose(y.max(), 2 * x[0, 367] - 1)   # center crop columns 144..367
print("crop + [-1,1] mapping OK", y.min(), y.max())

In [ ]:
model, _ = build_model(ModelConfig(name="resnet50_regression", dropout_p=0.5, bn_eps=1.001e-5))
eps = {m.eps for m in model.modules() if isinstance(m, torch.nn.BatchNorm2d)}
print("BatchNorm eps in model:", eps)   # one value, fixed at construction
t = torch.from_numpy(y).unsqueeze(0).unsqueeze(0)
print("model input shape", tuple(t.shape), "-> replicated to 3 channels inside forward")

In [ ]:
USE_REAL_IMAGE = False
if USE_REAL_IMAGE:
    from PIL import Image
    p = sorted(Path("data/raw/ldct_iqac/train/image").glob("*.tif"))[0]
    im = Image.open(p); print(p.name, im.mode, im.size)
    arr = np.array(im, dtype=np.float32); print("raw range", arr.min(), arr.max())
    out = preprocess_ct_image(arr, 224); print("processed range", out.min(), out.max(), out.dtype, out.shape)